# Redrob Candidate Ranking — Sandbox

**Hackathon sandbox** for the Redrob Intelligent Candidate Discovery & Ranking Challenge.

This notebook:
1. Installs dependencies
2. Clones the GitHub repo (update `REPO_URL` below)
3. Accepts up to 100 candidates as JSON
4. Runs the full ranking pipeline end-to-end
5. Displays results and downloads the submission CSV

**Runtime:** ~60-90 seconds for 50 candidates on Colab CPU free tier.

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers bm25s pyarrow pandas numpy scikit-learn

## Step 2 — Clone the repo

Update `REPO_URL` to your GitHub repo URL.

In [ ]:
import os

REPO_URL = "https://github.com/FILL_USERNAME/FILL_REPO_NAME.git"  # <-- UPDATE THIS

if not os.path.exists("repo"):
    os.system(f"git clone {REPO_URL} repo")
    print("Cloned.")
else:
    print("Repo already cloned.")

os.chdir("repo")
print("Working directory:", os.getcwd())
print("Files:", os.listdir("."))

## Step 3 — Upload candidates JSON

Upload a JSON file containing an array of up to 100 candidate objects.  
Or skip this cell to use the pre-loaded `data/sample_candidates.json` (50 candidates).

In [ ]:
from google.colab import files
import json

USE_SAMPLE = True   # Set False to upload your own file

if USE_SAMPLE:
    with open("data/sample_candidates.json", "r") as f:
        candidates = json.load(f)
    print(f"Using sample data: {len(candidates)} candidates")
else:
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    candidates = json.loads(uploaded[filename])
    if isinstance(candidates, dict):
        candidates = [candidates]
    candidates = candidates[:100]
    print(f"Uploaded: {len(candidates)} candidates")

## Step 4 — Run the full ranking pipeline

In [ ]:
import sys, time
sys.path.insert(0, ".")

from src.retrieval import score_sample
from src.scoring import score_batch
from src.reasoning import generate_reasoning, validate_no_hallucination

n = len(candidates)
print(f"Running pipeline on {n} candidates...")

# Step A: semantic retrieval (embed + BM25 + RRF)
t0 = time.time()
semantic_fits = score_sample(candidates, show_progress=True)
print(f"Semantic fits: {time.time()-t0:.1f}s")

# Step B: composite scoring
t0 = time.time()
results = score_batch(candidates, semantic_fits=semantic_fits)
print(f"Scoring: {time.time()-t0:.1f}s")

# Step C: build output
by_id = {c["candidate_id"]: c for c in candidates}
actual_n = min(100, len(results))
output_rows = []

for rank_idx, r in enumerate(results[:actual_n], 1):
    cid = r["candidate_id"]
    c   = by_id[cid]
    reasoning = generate_reasoning(rank_idx, c, r)
    violations = validate_no_hallucination(reasoning, c)
    if violations:
        print(f"  WARNING hallucination rank {rank_idx}: {violations}")
    output_rows.append({
        "candidate_id": cid,
        "rank"        : rank_idx,
        "score"       : round(r["final_score"], 5),
        "reasoning"   : reasoning,
        # Extra columns for display (not in final CSV)
        "_title"      : r["current_title"],
        "_yoe"        : r["yoe"],
        "_semantic"   : round(r["semantic_fit"], 3),
        "_career"     : round(r["career_quality"], 3),
        "_musthave"   : round(r["musthave_evidence"], 3),
        "_avail"      : r["availability"],
        "_penalty"    : round(r["penalty"], 3),
    })

print(f"\nDone. {len(output_rows)} candidates ranked.")

## Step 5 — View results

In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "rank"     : r["rank"],
    "id"       : r["candidate_id"],
    "title"    : r["_title"],
    "yoe"      : r["_yoe"],
    "score"    : r["score"],
    "semantic" : r["_semantic"],
    "career"   : r["_career"],
    "musthave" : r["_musthave"],
    "avail"    : r["_avail"],
    "penalty"  : r["_penalty"],
} for r in output_rows])

pd.set_option("display.max_colwidth", 35)
pd.set_option("display.max_rows", 30)
print("=== TOP 20 ===")
print(df.head(20).to_string(index=False))

In [ ]:
print("=== REASONING (top 10) ===")
for r in output_rows[:10]:
    print(f"\n#{r['rank']} {r['candidate_id']} (score={r['score']})")
    print(f"  {r['reasoning']}")

## Step 6 — Validate and download submission CSV

In [ ]:
import csv

csv_path = "/content/submission.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["candidate_id","rank","score","reasoning"])
    writer.writeheader()
    for r in output_rows:
        writer.writerow({
            "candidate_id": r["candidate_id"],
            "rank"        : r["rank"],
            "score"       : r["score"],
            "reasoning"   : r["reasoning"],
        })

# Basic validation
ids    = [r["candidate_id"] for r in output_rows]
scores = [r["score"] for r in output_rows]
ranks  = [r["rank"]  for r in output_rows]

checks = {
    f"{len(output_rows)} rows (expect 100 or all if <100 candidates)": True,
    "Unique candidate_ids"                : len(set(ids)) == len(ids),
    "Score non-increasing"                : all(scores[i] >= scores[i+1]-1e-9 for i in range(len(scores)-1)),
    "All reasoning filled"                : all(r["reasoning"].strip() for r in output_rows),
    "Ranks sequential from 1"             : sorted(ranks) == list(range(1, len(output_rows)+1)),
    "Encoding clean (no non-ASCII)"       : all(ord(ch) <= 127 for ch in open(csv_path).read()),
}

print("=== VALIDATION ===")
all_pass = True
for check, ok in checks.items():
    status = "PASS" if ok else "FAIL"
    print(f"  {status}  {check}")
    if not ok:
        all_pass = False

print("\nRESULT:", "PASSED" if all_pass else "FAILED")
print(f"CSV written to: {csv_path}")

In [ ]:
from google.colab import files
files.download(csv_path)
print("Download started.")

## Optional — Explain a specific candidate's score

In [ ]:
from src.scoring import explain

EXPLAIN_ID = "CAND_0000031"   # <-- change to any candidate_id

by_id = {c["candidate_id"]: c for c in candidates}
c = by_id.get(EXPLAIN_ID)

if c:
    sem = semantic_fits.get(EXPLAIN_ID, 0.0)
    explain(c, semantic_fit=sem)
else:
    print(f"{EXPLAIN_ID} not found in this batch")